In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import Subset, DataLoader

In [ ]:
if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device('cpu')
device

device(type='cuda')

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False

In [ ]:
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [ ]:
train_data = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_data = CIFAR10(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
best_accuracy = 0.0

for epoch in range(15):
    model.train()
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        pred = model(batch_X)
        loss = loss_fn(pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 1 == 0:
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                logits = model(batch_X)
                predictions = torch.argmax(logits, dim=1)
                correct += (predictions == batch_y).sum().item()
                total += batch_y.size(0)

        accuracy = correct / total
        print(f'Эпоха {epoch + 1}, ошибка {loss.item():.4f}, accuracy {accuracy:.2%}')

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), 'best_model.pt')

print(f'Лучший accuracy: {best_accuracy:.2%}')

Эпоха 1, ошибка 0.5735, accuracy 77.43%
Эпоха 2, ошибка 2.0936, accuracy 76.77%
Эпоха 3, ошибка 1.2878, accuracy 77.28%
Эпоха 4, ошибка 2.4873, accuracy 71.75%
Эпоха 5, ошибка 1.3662, accuracy 75.18%
Эпоха 6, ошибка 2.4994, accuracy 76.16%
Эпоха 7, ошибка 1.4368, accuracy 78.21%
Эпоха 8, ошибка 0.4727, accuracy 77.09%
Эпоха 9, ошибка 1.1707, accuracy 77.01%
Эпоха 10, ошибка 1.8997, accuracy 74.87%
Эпоха 11, ошибка 1.7173, accuracy 76.10%
Эпоха 12, ошибка 0.3259, accuracy 75.65%
Эпоха 13, ошибка 1.6125, accuracy 75.97%
Эпоха 14, ошибка 0.8390, accuracy 79.42%
Эпоха 15, ошибка 1.7751, accuracy 78.36%
Лучший accuracy: 79.42%


In [ ]:
from google.colab import files
files.download('best_model.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>